# TP05 : ACP, Arbres de Décision, Méthodes d'Ensemble et Pipelines sklearn
# 🔑 CORRECTION — Réservée à l'enseignant

> ⚠️ Ce fichier contient les solutions complètes. Ne pas distribuer aux étudiants.

## Organisation du TP

1. **ACP sur MNIST — Réduction de dimensionnalité et visualisation**
2. **Régression sur Composantes Principales (PCR)**
3. **Arbres de Décision — Classification sur MNIST**
4. **Méthodes d'ensemble — Random Forest et AdaBoost**
5. **⭐ Bonus — Pipeline complet et GridSearchCV**

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import time

from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    confusion_matrix, classification_report,
    mean_squared_error, accuracy_score
)

import warnings
warnings.filterwarnings('ignore')
np.random.seed(42)

In [ ]:
# Chargement MNIST
print("Chargement de MNIST...")
mnist = fetch_openml('mnist_784', version=1, as_frame=False, parser='auto')
X_full, y_full = mnist.data, mnist.target.astype(int)

idx = np.random.choice(len(X_full), size=10000, replace=False)
X, y = X_full[idx], y_full[idx]
X = X / 255.0

print(f"Forme de X : {X.shape}")

fig, axes = plt.subplots(2, 10, figsize=(15, 3))
for i, ax in enumerate(axes.ravel()):
    ax.imshow(X[i].reshape(28, 28), cmap='gray_r')
    ax.set_title(str(y[i]), fontsize=8)
    ax.axis('off')
plt.suptitle("Exemples MNIST", y=1.02)
plt.tight_layout()
plt.show()

---
# Partie 1 — ACP sur MNIST
## ✅ CORRECTION

In [ ]:
# ✅ Étape 1 — Standardisation et ACP complète
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)   # fit uniquement sur train !
X_test_sc  = scaler.transform(X_test)

pca_full = PCA()   # n_components=None → toutes les composantes
X_train_pca_full = pca_full.fit_transform(X_train_sc)

print(f"Nombre de composantes : {pca_full.n_components_}")
print(f"Variance expliquée par les 5 premières composantes : {pca_full.explained_variance_ratio_[:5].round(3)}")

In [ ]:
# ✅ Étape 2 — Scree plot et variance cumulée
cumvar = np.cumsum(pca_full.explained_variance_ratio_)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].plot(pca_full.explained_variance_ratio_[:100], 'o-', color='steelblue', markersize=3)
axes[0].set_xlabel("Composante principale")
axes[0].set_ylabel("Variance expliquée")
axes[0].set_title("Scree plot (100 premières composantes)")

axes[1].plot(cumvar, color='tomato')
axes[1].axhline(0.90, color='navy',  linestyle='--', label='90%')
axes[1].axhline(0.95, color='green', linestyle='--', label='95%')
axes[1].set_xlabel("Nombre de composantes")
axes[1].set_ylabel("Variance cumulée")
axes[1].set_title("Variance expliquée cumulée")
axes[1].legend()

plt.tight_layout()
plt.show()

n_90 = np.searchsorted(cumvar, 0.90) + 1
n_95 = np.searchsorted(cumvar, 0.95) + 1
print(f"Composantes pour 90% de variance : {n_90}")
print(f"Composantes pour 95% de variance : {n_95}")
print(f"Rapport de compression 90% : {n_90}/{X_train_sc.shape[1]} = {n_90/X_train_sc.shape[1]:.2%}")

### 🔑 Réponse attendue

- ~87 composantes suffisent pour 90% de la variance (à la place de 784) → ratio de compression ~11%
- ~120–150 pour 95%
- La chute rapide du scree plot indique que les images MNIST ont une forte structure sous-jacente : les pixels proches sont très corrélés (images naturelles ont des régions lisses).

In [ ]:
# ✅ Étape 3 — Eigendigits
fig, axes = plt.subplots(2, 10, figsize=(15, 3))

for i, ax in enumerate(axes.ravel()):
    comp = pca_full.components_[i].reshape(28, 28)
    ax.imshow(comp, cmap='RdBu_r')
    ax.set_title(f"PC{i+1}", fontsize=8)
    ax.axis('off')

plt.suptitle("Les 20 premières composantes principales — 'eigendigits'", y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ✅ Étape 4 — Reconstruction
n_components_list = [1, 10, 30, 50, 100, 200, 784]
fig, axes = plt.subplots(1, len(n_components_list), figsize=(16, 3))
sample = X_test_sc[0:1]   # première image du test

for ax, k in zip(axes, n_components_list):
    pca_k = PCA(n_components=k)
    pca_k.fit(X_train_sc)
    recon = pca_k.inverse_transform(pca_k.transform(sample))
    ax.imshow(recon.reshape(28, 28), cmap='gray_r')
    ax.set_title(f"k={k}", fontsize=9)
    ax.axis('off')

plt.suptitle("Reconstruction MNIST selon le nombre de composantes ACP", y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ✅ Étape 5 — Visualisation 2D
pca_2d = PCA(n_components=2)
X_train_2d = pca_2d.fit_transform(X_train_sc)

plt.figure(figsize=(9, 7))
scatter = plt.scatter(X_train_2d[:, 0], X_train_2d[:, 1],
                      c=y_train, cmap='tab10', alpha=0.4, s=8)
plt.colorbar(scatter, label='Chiffre')
plt.title("Projection ACP 2D — MNIST (coloré par classe)")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.tight_layout()
plt.show()

### 🔑 Réponse attendue

- Les classes ne sont **pas clairement séparées** en 2D — 2 composantes ne suffisent pas pour MNIST.
- On voit cependant des **tendances** : le 1 (chiffre fin) est souvent à l'écart, les 4/9/7 se chevauchent.
- Cela illustre que la classification MNIST nécessite beaucoup plus de dimensions pour être performante.

---
# Partie 2 — PCR
## ✅ CORRECTION

In [ ]:
# ✅ Étape 1 — Pipeline PCR et courbe MSE
n_components_range = [1, 5, 10, 20, 30, 50, 75, 100, 150, 200]
mse_train_list, mse_test_list = [], []

for k in n_components_range:
    pcr = Pipeline([
        ('scaler', StandardScaler()),
        ('pca',    PCA(n_components=k)),
        ('reg',    LinearRegression())
    ])
    pcr.fit(X_train, y_train)
    mse_train_list.append(mean_squared_error(y_train, pcr.predict(X_train)))
    mse_test_list.append(mean_squared_error(y_test,  pcr.predict(X_test)))

plt.figure(figsize=(9, 4))
plt.plot(n_components_range, mse_train_list, 'o-', label='Train MSE', color='steelblue')
plt.plot(n_components_range, mse_test_list,  's--', label='Test MSE',  color='tomato')
plt.xlabel("Nombre de composantes ACP (k)")
plt.ylabel("MSE")
plt.title("PCR — MSE selon le nombre de composantes")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ✅ Étape 2 — Sélection par CV
k_candidates = [10, 30, 50, 75, 100]
cv_mse = []

for k in k_candidates:
    pcr = Pipeline([
        ('scaler', StandardScaler()),
        ('pca',    PCA(n_components=k)),
        ('reg',    LinearRegression())
    ])
    scores = cross_val_score(pcr, X_train, y_train, cv=5, scoring='neg_mean_squared_error')
    cv_mse.append(-scores.mean())

best_k = k_candidates[np.argmin(cv_mse)]
print(f"Meilleur k (CV) : {best_k}  —  MSE CV = {min(cv_mse):.4f}")

plt.figure(figsize=(7, 4))
plt.plot(k_candidates, cv_mse, 'o-', color='purple')
plt.axvline(best_k, color='red', linestyle='--', label=f'k optimal = {best_k}')
plt.xlabel("Nombre de composantes")
plt.ylabel("MSE (CV 5-fold)")
plt.title("Sélection de k par validation croisée")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ✅ Étape 3 — OLS vs PCR
ols = Pipeline([('scaler', StandardScaler()), ('reg', LinearRegression())])
ols.fit(X_train, y_train)
ols_mse  = mean_squared_error(y_test, ols.predict(X_test))
ols_acc  = accuracy_score(y_test, np.clip(np.round(ols.predict(X_test)), 0, 9).astype(int))

pcr_best = Pipeline([
    ('scaler', StandardScaler()),
    ('pca',    PCA(n_components=best_k)),
    ('reg',    LinearRegression())
])
pcr_best.fit(X_train, y_train)
pcr_mse = mean_squared_error(y_test, pcr_best.predict(X_test))
pcr_acc = accuracy_score(y_test, np.clip(np.round(pcr_best.predict(X_test)), 0, 9).astype(int))

print("=== Comparaison OLS vs PCR ===")
print(f"OLS brut     — MSE test : {ols_mse:.4f}  | Accuracy (arrondi) : {ols_acc:.4f}")
print(f"PCR (k={best_k}) — MSE test : {pcr_mse:.4f}  | Accuracy (arrondi) : {pcr_acc:.4f}")

### 🔑 Réponse attendue

- La régression linéaire arrondie atteint ~60–70% d'accuracy — correct mais bien en dessous des méthodes de classification.
- La PCR réduit légèrement le MSE par rapport à l'OLS brut (moins de surapprentissage).
- La régression linéaire n'est pas conçue pour la classification : elle prédit des valeurs continues (ex. 4.7) et ne modélise pas les probabilités de classe. La régression logistique ou les arbres sont plus adaptés.

---
# Partie 3 — Arbres de Décision
## ✅ CORRECTION

In [ ]:
# ✅ Étape 1 — Entropie et gain d'information
def entropy(y):
    classes, counts = np.unique(y, return_counts=True)
    p = counts / counts.sum()
    return -np.sum(p * np.log2(p + 1e-12))

def gini(y):
    classes, counts = np.unique(y, return_counts=True)
    p = counts / counts.sum()
    return 1 - np.sum(p**2)

def information_gain(y_parent, y_left, y_right):
    n = len(y_parent)
    return entropy(y_parent) - (len(y_left)/n) * entropy(y_left) - (len(y_right)/n) * entropy(y_right)

y_parent = np.array([0,0,0,0,1,1,1,1])
y_left   = np.array([0,0,0,1])
y_right  = np.array([0,1,1,1])

print(f"Entropie parent  : {entropy(y_parent):.4f}  (attendu ≈ 1.0)")
print(f"Gini parent      : {gini(y_parent):.4f}  (attendu = 0.5)")
print(f"Gain information : {information_gain(y_parent, y_left, y_right):.4f}")

In [ ]:
# ✅ Étape 2 — Arbre binaire 0 vs 1
mask_train = np.isin(y_train, [0, 1])
mask_test  = np.isin(y_test,  [0, 1])
Xb_train, yb_train = X_train_sc[mask_train], y_train[mask_train]
Xb_test,  yb_test  = X_test_sc[mask_test],   y_test[mask_test]

tree_small = DecisionTreeClassifier(max_depth=3, criterion='gini', random_state=42)
tree_small.fit(Xb_train, yb_train)

fig, ax = plt.subplots(figsize=(20, 6))
plot_tree(tree_small, max_depth=3, filled=True,
          feature_names=[f'px_{i}' for i in range(784)],
          class_names=['0','1'], ax=ax, fontsize=8)
plt.title("Arbre de décision — MNIST binaire (0 vs 1), max_depth=3")
plt.tight_layout()
plt.show()

print(f"Accuracy train : {tree_small.score(Xb_train, yb_train):.4f}")
print(f"Accuracy test  : {tree_small.score(Xb_test, yb_test):.4f}")

In [ ]:
# ✅ Étape 3 — Surapprentissage selon la profondeur
depths = [1, 3, 5, 8, 10, 15, None]
acc_train_tree, acc_test_tree = [], []

for d in depths:
    tree = DecisionTreeClassifier(max_depth=d, random_state=42)
    tree.fit(X_train_sc, y_train)
    acc_train_tree.append(tree.score(X_train_sc, y_train))
    acc_test_tree.append(tree.score(X_test_sc, y_test))

depths_plot = [d if d is not None else 20 for d in depths]
labels = [str(d) if d is not None else 'None\n(complet)' for d in depths]

plt.figure(figsize=(9, 4))
plt.plot(depths_plot, acc_train_tree, 'o-', label='Train', color='steelblue')
plt.plot(depths_plot, acc_test_tree,  's--', label='Test',  color='tomato')
plt.xticks(depths_plot, labels)
plt.xlabel("max_depth")
plt.ylabel("Accuracy")
plt.title("Arbre de décision — Surapprentissage selon la profondeur")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ✅ Étape 4 — Pixels bruts vs ACP
pca_50 = PCA(n_components=50)
X_train_50 = pca_50.fit_transform(X_train_sc)
X_test_50  = pca_50.transform(X_test_sc)

# Arbre sur pixels bruts
t0 = time.time()
tree_raw = DecisionTreeClassifier(max_depth=10, random_state=42)
tree_raw.fit(X_train_sc, y_train)
acc_raw = tree_raw.score(X_test_sc, y_test)
t_raw = time.time() - t0

# Arbre sur ACP
t0 = time.time()
tree_pca = DecisionTreeClassifier(max_depth=10, random_state=42)
tree_pca.fit(X_train_50, y_train)
acc_pca = tree_pca.score(X_test_50, y_test)
t_pca = time.time() - t0

print("=== Arbre de décision (max_depth=10) ===")
print(f"Pixels bruts (784 feat.) — Accuracy test : {acc_raw:.4f}  | Temps : {t_raw:.2f}s")
print(f"ACP 50 composantes       — Accuracy test : {acc_pca:.4f}  | Temps : {t_pca:.2f}s")

### 🔑 Réponse attendue

- Le surapprentissage apparaît dès **max_depth ≥ 8–10** : accuracy train → 1.0 mais test stagne ou baisse.
- L'ACP réduit le temps d'entraînement (moins de features à évaluer) et peut légèrement améliorer l'accuracy (moins de bruit).
- Limite fondamentale d'un seul arbre : **haute variance** — très sensible aux données d'entraînement, ce que les ensembles corrigent.

---
# Partie 4 — Méthodes d'Ensemble
## ✅ CORRECTION

In [ ]:
# ✅ Étape 1 — Random Forest
t0 = time.time()
rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X_train_50, y_train)
acc_rf_train = rf.score(X_train_50, y_train)
acc_rf_test  = rf.score(X_test_50, y_test)
t_rf = time.time() - t0

print(f"Random Forest — Accuracy train : {acc_rf_train:.4f}  | test : {acc_rf_test:.4f}  | Temps : {t_rf:.2f}s")

In [ ]:
# ✅ Étape 2 — Effet du nombre d'arbres
n_estimators_list = [1, 5, 10, 25, 50, 100, 200]
rf_acc_test = []

for n in n_estimators_list:
    rf_n = RandomForestClassifier(n_estimators=n, random_state=42, n_jobs=-1)
    rf_n.fit(X_train_50, y_train)
    rf_acc_test.append(rf_n.score(X_test_50, y_test))

plt.figure(figsize=(8, 4))
plt.plot(n_estimators_list, rf_acc_test, 'o-', color='green')
plt.xlabel("Nombre d'arbres (n_estimators)")
plt.ylabel("Accuracy test")
plt.title("Random Forest — Accuracy selon le nombre d'arbres")
plt.tight_layout()
plt.show()

In [ ]:
# ✅ Étape 3 — AdaBoost
t0 = time.time()
ada = AdaBoostClassifier(n_estimators=100, random_state=42, algorithm='SAMME')
ada.fit(X_train_50, y_train)
acc_ada_train = ada.score(X_train_50, y_train)
acc_ada_test  = ada.score(X_test_50, y_test)
t_ada = time.time() - t0

print(f"AdaBoost — Accuracy train : {acc_ada_train:.4f}  | test : {acc_ada_test:.4f}  | Temps : {t_ada:.2f}s")

In [ ]:
# ✅ Étape 4 — Tableau comparatif
import pandas as pd

results = pd.DataFrame({
    'Modèle': ['Arbre (max_depth=10, raw)', 'Arbre (max_depth=10, ACP-50)',
               'Random Forest (100 arbres, ACP-50)', 'AdaBoost (100 estim., ACP-50)'],
    'Accuracy Test': [acc_raw, acc_pca, acc_rf_test, acc_ada_test],
    'Temps (s)':     [t_raw, t_pca, t_rf, t_ada]
})
results['Accuracy Test'] = results['Accuracy Test'].round(4)
results['Temps (s)']     = results['Temps (s)'].round(2)
print(results.to_string(index=False))

# Matrice de confusion du Random Forest (généralement le meilleur)
y_pred_rf = rf.predict(X_test_50)
cm = confusion_matrix(y_test, y_pred_rf)
plt.figure(figsize=(9, 7))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=range(10), yticklabels=range(10))
plt.title("Matrice de confusion — Random Forest (test)")
plt.xlabel("Prédit")
plt.ylabel("Réel")
plt.tight_layout()
plt.show()

print(classification_report(y_test, y_pred_rf))

### 🔑 Réponse attendue

- **Random Forest** offre généralement le meilleur compromis : accuracy ~93–95% sur ACP-50, temps raisonnable.
- Chiffres souvent confondus : **4/9**, **3/5**, **7/9** — similitudes morphologiques.
- RF > arbre seul car la moyenne de T arbres décorrélés réduit la **variance** sans augmenter le biais.
- Bagging = parallèle (bootstrap indépendants) / Boosting = séquentiel (correction des erreurs précédentes).

---
# Partie 5 — ⭐ Bonus : Pipeline + GridSearchCV
## ✅ CORRECTION

In [ ]:
# ✅ Étape 1 — Construction du Pipeline
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('pca',    PCA()),
    ('clf',    DecisionTreeClassifier(random_state=42))
])

print(pipeline)

In [ ]:
# ✅ Étape 2 — GridSearchCV sur ACP + Arbre
# Sous-ensemble de 2000 exemples pour la rapidité
X_gs, _, y_gs, _ = train_test_split(X_train_sc, y_train, train_size=2000, random_state=42, stratify=y_train)

param_grid = {
    'pca__n_components': [20, 50, 100],
    'clf__max_depth':    [5, 10, 15],
    'clf__criterion':   ['gini', 'entropy']
}

grid_search = GridSearchCV(
    pipeline, param_grid,
    cv=3, scoring='accuracy', n_jobs=-1, verbose=1
)
grid_search.fit(X_gs, y_gs)

print(f"Meilleurs paramètres : {grid_search.best_params_}")
print(f"Meilleur score CV    : {grid_search.best_score_:.4f}")

best_pipe = grid_search.best_estimator_
print(f"Accuracy test (meilleur pipeline) : {best_pipe.score(X_test_sc, y_test):.4f}")

In [ ]:
# ✅ Étape 3 — Exploration avec Random Forest (contrainte : n_components <= 50)
pipeline_rf = Pipeline([
    ('scaler', StandardScaler()),
    ('pca',    PCA()),
    ('clf',    RandomForestClassifier(random_state=42, n_jobs=-1))
])

param_grid_rf = {
    'pca__n_components': [20, 30, 50],    # contrainte : <= 50
    'clf__n_estimators': [50, 100],
    'clf__max_depth':    [None, 15]
}

grid_rf = GridSearchCV(
    pipeline_rf, param_grid_rf,
    cv=3, scoring='accuracy', n_jobs=-1, verbose=1
)
grid_rf.fit(X_gs, y_gs)

print(f"\nMeilleurs paramètres RF : {grid_rf.best_params_}")
print(f"Meilleur score CV       : {grid_rf.best_score_:.4f}")
print(f"Accuracy test           : {grid_rf.best_estimator_.score(X_test_sc, y_test):.4f}")

### 🔑 Réponse attendue

- Avec 50 composantes + RF 100 arbres → accuracy test ~93–95%, meilleure combinaison.
- GridSearchCV teste **toutes les combinaisons** × **cv folds** → peut être coûteux : ici 3×3×2 × 3 = 54 fits pour l'arbre.
- Alternatives : **RandomizedSearchCV** (échantillonnage aléatoire de la grille), **Optuna** ou **HalvingGridSearchCV** pour de plus grands espaces.

---

## Résumé — Points Clés

| Concept | Point clé |
|---|---|
| ACP | Réduction de 784 → quelques dizaines de composantes sans perte majeure d'information |
| Eigendigits | Les composantes principales capturent les modes de variation visuels |
| PCR | ACP + régression — contrôle la multicolinéarité, k choisi par CV |
| Arbre de décision | Interprétable mais sur-apprend facilement (max_depth critique) |
| Random Forest | Bagging d'arbres → variance réduite, plus robuste qu'un seul arbre |
| AdaBoost | Boosting séquentiel → biais réduit, sensible aux outliers |
| Pipeline sklearn | Chaîne reproductible et compatible avec GridSearchCV |
| GridSearchCV | Recherche automatique des hyperparamètres optimaux par validation croisée |